# Perbandingan Model: Deep Learning (BiLSTM) vs Logistic Regression

Notebook ini membandingkan kedua model secara **adil** menggunakan simulasi kondisi produksi:
- Kedua model dimuat dari disk
- Menggunakan data uji yang sama dan independen
- Metrik: Accuracy, F1-Score, Kecepatan Load, Kecepatan Inferensi

## LANGKAH 1 — Import Library dan Konfigurasi Path

In [1]:
from __future__ import annotations

import json
import time
import joblib
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, f1_score

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json

BASE_DIR = Path(".").resolve()
DL_DIR   = (BASE_DIR.parent / "chatbot_models" / "deep_learning").resolve()
LR_DIR   = (BASE_DIR.parent / "chatbot_models" / "logistic_regression").resolve()
MAX_LEN  = 20

print("Deep Learning dir :", DL_DIR)
print("Logistic Reg dir  :", LR_DIR)
print("TensorFlow versi  :", tf.__version__)

Deep Learning dir : C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\deep_learning
Logistic Reg dir  : C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\logistic_regression
TensorFlow versi  : 2.21.0


## LANGKAH 2 — Data Uji Independen

Kalimat uji ini tidak ada di TRAINING_SAMPLES kedua model.

In [2]:
TEST_SAMPLES = [
    # 1. FAKTOR_CHURN
    ("apa penyebab utama pelanggan berhenti",                    "FAKTOR_CHURN"),
    ("apa yang membuat orang tidak mau lanjut berlangganan",     "FAKTOR_CHURN"),
    ("variabel terpenting penentu churn",                        "FAKTOR_CHURN"),
    ("kok bisa sih pada unsubscribe", "FAKTOR_CHURN"),
    ("alasan dominan orang pada cancel apa ya", "FAKTOR_CHURN"),
    
    # 2. VIP_RISK
    ("pelanggan enterprise mana yang hampir pergi",              "VIP_RISK"),
    ("siapa customer high value yang mau berhenti",              "VIP_RISK"),
    ("klien kakap yang mau cabut siapa aja", "VIP_RISK"),
    ("ada gak sultan yang mau putus kontrak", "VIP_RISK"),
    
    # 3. JUMLAH_RISIKO_TINGGI
    ("ada berapa orang yang masuk kategori berbahaya",           "JUMLAH_RISIKO_TINGGI"),
    ("berapa customer yang probabilitas churn-nya tinggi",       "JUMLAH_RISIKO_TINGGI"),
    ("hitungin dong yang masuk zona merah", "JUMLAH_RISIKO_TINGGI"),
    ("total user yg mau kabur brp", "JUMLAH_RISIKO_TINGGI"),
    
    # 4. ANALISIS_PELANGGAN
    ("tunjukkan data lengkap C-0015",                            "ANALISIS_PELANGGAN"),
    ("apa status pelanggan C-0009",                              "ANALISIS_PELANGGAN"),
    ("kepoin si C-0123 dong", "ANALISIS_PELANGGAN"),
    ("kasih liat profil C-0050", "ANALISIS_PELANGGAN"),
    
    # 5. STRATEGI_RETENSI
    ("apa langkah konkret untuk menekan angka churn",            "STRATEGI_RETENSI"),
    ("kebijakan apa yang bisa kita ambil untuk retensi",         "STRATEGI_RETENSI"),
    ("gimana cara nahan user biar ga pergi", "STRATEGI_RETENSI"),
    ("tips jitu kurangin churn dong", "STRATEGI_RETENSI"),
    
    # 6. DRAF_EMAIL
    ("susunkan email penawaran buat C-0003",                     "DRAF_EMAIL"),
    ("buatin saya template email untuk customer yang mau pergi", "DRAF_EMAIL"),
    ("tolong ketikin pesan buat C-0021", "DRAF_EMAIL"),
    ("bikin email diskon untuk C-0010", "DRAF_EMAIL"),
    
    # 7. GREETING
    ("good morning",                                             "GREETING"),
    ("hei apa kabar bot",                                        "GREETING"),
    ("bro halo", "GREETING"),
    ("pagi min", "GREETING"),
    ("lagi ngapain bot", "GREETING"),
    
    # 8. TREN_CHURN
    ("bagaimana perubahan churn dari bulan ke bulan",            "TREN_CHURN"),
    ("apakah angka churn kita membaik",                          "TREN_CHURN"),
    ("grafik cancel bulan ini gimana", "TREN_CHURN"),
    ("lagi naik apa turun nih yang churn", "TREN_CHURN"),
    
    # 9. SEGMEN_ANALISIS
    ("mana yang lebih banyak churn antara starter dan enterprise", "SEGMEN_ANALISIS"),
    ("persentase churn tiap kelompok pelanggan",                 "SEGMEN_ANALISIS"),
    ("plan bulanan vs tahunan banyakan mana yang cabut", "SEGMEN_ANALISIS"),
    ("komparasi churn tiap segmen", "SEGMEN_ANALISIS"),
    
    # 10. MODEL_INFO
    ("teknologi apa yang dipakai untuk prediksi churn ini",      "MODEL_INFO"),
    ("seberapa akurat sistem prediksi ini",                      "MODEL_INFO"),
    ("ini AI nya pakai apa", "MODEL_INFO"),
    ("cara kerjanya gmn", "MODEL_INFO"),
    
    # 11. METRIK_OVERVIEW
    ("kasih saya snapshot kondisi bisnis hari ini",              "METRIK_OVERVIEW"),
    ("seperti apa situasi churn kita secara keseluruhan",        "METRIK_OVERVIEW"),
    ("update dong hari ini", "METRIK_OVERVIEW"),
    ("kabar bisnis hari ini gimana", "METRIK_OVERVIEW"),
]

test_texts  = [t for t, _ in TEST_SAMPLES]
test_labels = [l for _, l in TEST_SAMPLES]

print(f"Total data uji  : {len(TEST_SAMPLES)} kalimat")
print(f"Intent tercakup : {sorted(set(test_labels))}")

Total data uji  : 46 kalimat
Intent tercakup : ['ANALISIS_PELANGGAN', 'DRAF_EMAIL', 'FAKTOR_CHURN', 'GREETING', 'JUMLAH_RISIKO_TINGGI', 'METRIK_OVERVIEW', 'MODEL_INFO', 'SEGMEN_ANALISIS', 'STRATEGI_RETENSI', 'TREN_CHURN', 'VIP_RISK']


## LANGKAH 3 — Load dan Evaluasi Logistic Regression dari Disk

In [3]:
t0 = time.time()
clf_lr = joblib.load(LR_DIR / "intent_model.pkl")
vec_lr = joblib.load(LR_DIR / "intent_vectorizer.pkl")
t_load_lr = time.time() - t0

X_te_lr = vec_lr.transform(test_texts)
t0 = time.time()
y_pred_lr = clf_lr.predict(X_te_lr)
t_infer_lr = (time.time() - t0) / len(test_texts) * 1000

lr_acc = accuracy_score(test_labels, y_pred_lr)
lr_f1  = f1_score(test_labels, y_pred_lr, average="weighted", zero_division=0)

print("=" * 55)
print("LOGISTIC REGRESSION (TF-IDF)")
print("=" * 55)
print(f"Waktu load model   : {t_load_lr:.4f} detik")
print(f"Waktu inferensi    : {t_infer_lr:.4f} ms/kalimat")
print(f"Accuracy           : {lr_acc:.2%}")
print(f"F1-Score (weighted): {lr_f1:.2%}")
print()
print(classification_report(test_labels, y_pred_lr, zero_division=0))

LOGISTIC REGRESSION (TF-IDF)
Waktu load model   : 0.0699 detik
Waktu inferensi    : 0.0112 ms/kalimat
Accuracy           : 84.78%
F1-Score (weighted): 84.85%

                      precision    recall  f1-score   support

  ANALISIS_PELANGGAN       1.00      0.75      0.86         4
          DRAF_EMAIL       1.00      1.00      1.00         4
        FAKTOR_CHURN       0.80      0.80      0.80         5
            GREETING       0.83      1.00      0.91         5
JUMLAH_RISIKO_TINGGI       0.75      0.75      0.75         4
     METRIK_OVERVIEW       0.67      1.00      0.80         4
          MODEL_INFO       1.00      0.75      0.86         4
     SEGMEN_ANALISIS       1.00      1.00      1.00         4
    STRATEGI_RETENSI       0.75      0.75      0.75         4
          TREN_CHURN       1.00      0.75      0.86         4
            VIP_RISK       0.75      0.75      0.75         4

            accuracy                           0.85        46
           macro avg       0.87  

## LANGKAH 4 — Load dan Evaluasi Deep Learning dari Disk

In [4]:
t0 = time.time()
_model_dl = load_model(DL_DIR / "intent_model.keras")
with open(DL_DIR / "intent_tokenizer.json", "r", encoding="utf-8") as f:
    _tok_dl = tokenizer_from_json(f.read())
with open(DL_DIR / "intent_label_map.json", "r", encoding="utf-8") as f:
    _lmap_dl = json.load(f)["idx_to_label"]
t_load_dl = time.time() - t0

seqs   = _tok_dl.texts_to_sequences(test_texts)
padded = pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")
t0 = time.time()
probs  = _model_dl.predict(padded, verbose=0)
t_infer_dl = (time.time() - t0) / len(test_texts) * 1000

y_pred_dl = [_lmap_dl[str(int(np.argmax(p)))] for p in probs]
dl_acc    = accuracy_score(test_labels, y_pred_dl)
dl_f1     = f1_score(test_labels, y_pred_dl, average="weighted", zero_division=0)

print("=" * 55)
print("DEEP LEARNING (BiLSTM + Word Embedding)")
print("=" * 55)
print(f"Waktu load model   : {t_load_dl:.4f} detik")
print(f"Waktu inferensi    : {t_infer_dl:.4f} ms/kalimat")
print(f"Accuracy           : {dl_acc:.2%}")
print(f"F1-Score (weighted): {dl_f1:.2%}")
print()
print(classification_report(test_labels, y_pred_dl, zero_division=0))

DEEP LEARNING (BiLSTM + Word Embedding)
Waktu load model   : 0.2157 detik
Waktu inferensi    : 10.6485 ms/kalimat
Accuracy           : 91.30%
F1-Score (weighted): 91.21%

                      precision    recall  f1-score   support

  ANALISIS_PELANGGAN       1.00      1.00      1.00         4
          DRAF_EMAIL       1.00      1.00      1.00         4
        FAKTOR_CHURN       1.00      0.80      0.89         5
            GREETING       0.83      1.00      0.91         5
JUMLAH_RISIKO_TINGGI       0.75      0.75      0.75         4
     METRIK_OVERVIEW       1.00      1.00      1.00         4
          MODEL_INFO       1.00      0.75      0.86         4
     SEGMEN_ANALISIS       1.00      1.00      1.00         4
    STRATEGI_RETENSI       0.80      1.00      0.89         4
          TREN_CHURN       0.80      1.00      0.89         4
            VIP_RISK       1.00      0.75      0.86         4

            accuracy                           0.91        46
           macro avg 

## LANGKAH 5 — Tabel Ringkasan dan Kesimpulan Final

In [5]:
lr_size_kb = (LR_DIR / "intent_model.pkl").stat().st_size / 1024
dl_size_kb = (DL_DIR / "intent_model.keras").stat().st_size / 1024

print("=" * 62)
print("RINGKASAN PERBANDINGAN FINAL")
print("=" * 62)
print(f"{'Metrik':<30} {'Logistic Reg':>14} {'Deep Learning':>14}")
print("-" * 62)
print(f"{'Accuracy':<30} {lr_acc:>13.2%} {dl_acc:>13.2%}")
print(f"{'F1-Score (weighted)':<30} {lr_f1:>13.2%} {dl_f1:>13.2%}")
print(f"{'Waktu Load Model (detik)':<30} {t_load_lr:>13.4f} {t_load_dl:>13.4f}")
print(f"{'Waktu Inferensi (ms/kal)':<30} {t_infer_lr:>13.4f} {t_infer_dl:>13.4f}")
print(f"{'Ukuran File Model (KB)':<30} {lr_size_kb:>13.1f} {dl_size_kb:>13.1f}")
print("=" * 62)

wins_lr = sum([
    lr_acc     >= dl_acc,
    lr_f1      >= dl_f1,
    t_load_lr  <= t_load_dl,
    t_infer_lr <= t_infer_dl,
    lr_size_kb <= dl_size_kb,
])
wins_dl = 5 - wins_lr

print()
print("PEMENANG TIAP KATEGORI:")
print(f"  Akurasi tertinggi    : {'Logistic Reg' if lr_acc     >= dl_acc     else 'Deep Learning'}")
print(f"  F1-Score tertinggi   : {'Logistic Reg' if lr_f1      >= dl_f1      else 'Deep Learning'}")
print(f"  Load tercepat        : {'Logistic Reg' if t_load_lr  <= t_load_dl  else 'Deep Learning'}")
print(f"  Inferensi tercepat   : {'Logistic Reg' if t_infer_lr <= t_infer_dl else 'Deep Learning'}")
print(f"  Model paling ringan  : {'Logistic Reg' if lr_size_kb <= dl_size_kb else 'Deep Learning'}")
print()
print(f"Skor : Logistic Reg {wins_lr}/5  vs  Deep Learning {wins_dl}/5")
print()
winner = "Logistic Regression" if wins_lr >= wins_dl else "Deep Learning (BiLSTM)"
print("=" * 62)
print(f"REKOMENDASI MODEL TERBAIK: {winner}")
print("=" * 62)

RINGKASAN PERBANDINGAN FINAL
Metrik                           Logistic Reg  Deep Learning
--------------------------------------------------------------
Accuracy                              84.78%        91.30%
F1-Score (weighted)                   84.85%        91.21%
Waktu Load Model (detik)              0.0699        0.2157
Waktu Inferensi (ms/kal)              0.0112       10.6485
Ukuran File Model (KB)                 548.5        1621.3

PEMENANG TIAP KATEGORI:
  Akurasi tertinggi    : Deep Learning
  F1-Score tertinggi   : Deep Learning
  Load tercepat        : Logistic Reg
  Inferensi tercepat   : Logistic Reg
  Model paling ringan  : Logistic Reg

Skor : Logistic Reg 3/5  vs  Deep Learning 2/5

REKOMENDASI MODEL TERBAIK: Logistic Regression
